# Train the gauge keypoint model on Kaggle

1. Add the dataset: **Add Input → Your Datasets → synth-2d** (the zip of `data/synthetic/synth_2d`, see docs/TRAINING.md).
2. Settings: **Accelerator GPU T4 x2**, **Internet on** (needs a phone-verified Kaggle account).
3. **Save Version → Save & Run All (Commit)** runs it in the background (up to 12 h). When it finishes, download `runs/pose/base/weights/best.pt` and `results.csv` from the Output tab.

In [ ]:
EPOCHS = 50
MODEL = "yolo11s-pose.pt"
BATCH = 32          # total over both GPUs
DEVICE = "0,1"      # "0" on a single P100/T4
NAME = "base"
FRACTION = 1.0      # e.g. 0.05 with EPOCHS = 1 for a quick check

In [ ]:
!pip install -q "ultralytics>=8.3"
!git clone -q https://github.com/uhyon/mv2npp-gauge-reading.git
%cd mv2npp-gauge-reading
!git log --oneline -1

In [ ]:
import glob, os
found = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
assert found, "dataset not attached: Add Input -> synth-2d"
DATA = os.path.dirname(found[0])
print(DATA, len(os.listdir(f"{DATA}/images/train")), "train images")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python scripts/train_pose.py --data {DATA} --model {MODEL} --epochs {EPOCHS} --batch {BATCH} \
    --device {DEVICE} --workers 4 --fraction {FRACTION} --project /kaggle/working/runs/pose --name {NAME}

In [ ]:
import pandas as pd
r = pd.read_csv(f"/kaggle/working/runs/pose/{NAME}/results.csv")
r.columns = r.columns.str.strip()
r[[c for c in r.columns if c.startswith(("epoch", "metrics/"))]].tail(10)

If the session stops early, attach this notebook's own output as an input and resume:
`!python scripts/train_pose.py --resume /kaggle/input/<this notebook output>/runs/pose/base/weights/last.pt`